# Adım 6: Makine Öğrenmesi + MLflow
**Beyda sorumluluğu** — `feature/ml-dashboard` branch

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, Imputer
from pyspark.ml.regression import (
    LinearRegression, DecisionTreeRegressor,
    RandomForestRegressor, GBTRegressor, GeneralizedLinearRegression,
)
from pyspark.ml.evaluation import RegressionEvaluator
import mlflow
import mlflow.spark
import pandas as pd
import json
import os

FEATURE_PATH = './delta_lake/features'
MODEL_PATH   = './mlflow_data/best_model'
os.makedirs('./mlflow_data', exist_ok=True)

FEATURE_COLS = [
    'temp_range', 'month', 'season_num', 'rolling_avg_7',
    'is_extreme_temp', 'prcp_category',
    'avg_wind_speed_kmh', 'avg_sea_level_pres_hpa', 'sunshine_total_min',
]
TARGET_COL = 'avg_temp_c'

def create_spark():
    return (
        SparkSession.builder
        .appName('ClimateML')
        .master('local[*]')
        .config('spark.sql.extensions',
                'io.delta.sql.DeltaSparkSessionExtension')
        .config('spark.sql.catalog.spark_catalog',
                'org.apache.spark.sql.delta.catalog.DeltaCatalog')
        .config('spark.jars.packages',
                'io.delta:delta-core_2.12:2.4.0')
        .getOrCreate()
    )

spark = create_spark()
spark.sparkContext.setLogLevel('WARN')
df = spark.read.format('delta').load(FEATURE_PATH)
df = df.dropna(subset=[TARGET_COL] + FEATURE_COLS)
print(f'[ML] Veri yuklendi: {df.count():,} kayit')
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
print(f'[ML] Egitim: {train_df.count():,}  |  Test: {test_df.count():,}')

26/05/12 14:54:11 WARN Utils: Your hostname, Canpolat-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 172.20.10.14 instead (on interface en0)
26/05/12 14:54:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/canpolat/.ivy2/cache
The jars for the packages stored in: /Users/canpolat/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a6853444-7a08-4824-98d7-702b62f97e75;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central


:: loading settings :: url = jar:file:/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 86ms :: artifacts dl 3ms
	:: modules in use:
	io.delta#delta-core_2.12;2.4.0 from central in [default]
	io.delta#delta-storage;2.4.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-a6853444-7a08-4824-98d7-702b62f97e75
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/3ms)
26/05/12 14:54:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-jav

[ML] Veri yuklendi: 704,208 kayit


[ML] Egitim: 563,797  |  Test: 140,411


In [2]:
def build_pipeline(model):
    imputer = Imputer(inputCols=FEATURE_COLS,
                     outputCols=[f'{c}_imp' for c in FEATURE_COLS])
    assembler = VectorAssembler(inputCols=[f'{c}_imp' for c in FEATURE_COLS],
                                outputCol='features')
    return Pipeline(stages=[imputer, assembler, model])

def evaluate(predictions):
    ev = RegressionEvaluator(labelCol=TARGET_COL, predictionCol='prediction')
    return {
        'rmse': round(ev.setMetricName('rmse').evaluate(predictions), 4),
        'mae':  round(ev.setMetricName('mae').evaluate(predictions), 4),
        'r2':   round(ev.setMetricName('r2').evaluate(predictions), 4),
    }

def get_feature_importance(fitted_model):
    try:
        stage = fitted_model.stages[-1]
        if hasattr(stage, 'featureImportances'):
            return dict(zip(FEATURE_COLS, stage.featureImportances.toArray().tolist()))
    except Exception:
        pass
    return {}

mlflow.set_tracking_uri('./mlruns')
mlflow.set_experiment('climate-temperature-prediction')
results = []

In [3]:
# lineer regresyon modeli
print('[ML] LinearRegression egitiliyor...')
with mlflow.start_run(run_name='LinearRegression'):
    model = LinearRegression(labelCol=TARGET_COL, featuresCol='features', maxIter=100)
    fitted = build_pipeline(model).fit(train_df)
    metrics = evaluate(fitted.transform(test_df))
    mlflow.log_param('model', 'LinearRegression')
    mlflow.log_param('maxIter', 100)
    mlflow.log_metrics(metrics)
    print(f'  RMSE={metrics["rmse"]}  MAE={metrics["mae"]}  R2={metrics["r2"]}')
    results.append({'model': 'LinearRegression', **metrics, 'feature_importance': {}, 'fitted': fitted})

# karar agaci
print('[ML] DecisionTree egitiliyor...')
with mlflow.start_run(run_name='DecisionTree'):
    model = DecisionTreeRegressor(labelCol=TARGET_COL, featuresCol='features', maxDepth=5)
    fitted = build_pipeline(model).fit(train_df)
    metrics = evaluate(fitted.transform(test_df))
    fi = get_feature_importance(fitted)
    mlflow.log_param('model', 'DecisionTree')
    mlflow.log_param('maxDepth', 5)
    mlflow.log_metrics(metrics)
    if fi: mlflow.log_dict(fi, 'feature_importance.json')
    print(f'  RMSE={metrics["rmse"]}  MAE={metrics["mae"]}  R2={metrics["r2"]}')
    results.append({'model': 'DecisionTree', **metrics, 'feature_importance': fi, 'fitted': fitted})

[ML] LinearRegression egitiliyor...


26/05/12 14:54:24 WARN Instrumentation: [cffd0c68] regParam is zero, which might cause numerical instability and overfitting.
26/05/12 14:54:26 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/05/12 14:54:26 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
26/05/12 14:54:27 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


  RMSE=2.9714  MAE=2.27  R2=0.9049
[ML] DecisionTree egitiliyor...


  RMSE=3.103  MAE=2.3579  R2=0.8963


## Random Forest + GBT + Generalized Linear Regression

In [4]:
# random forest - daha iyi genelleme bekliyoruz
print('[ML] RandomForest egitiliyor...')
with mlflow.start_run(run_name='RandomForest'):
    model = RandomForestRegressor(labelCol=TARGET_COL, featuresCol='features', numTrees=50, seed=42)
    fitted = build_pipeline(model).fit(train_df)
    metrics = evaluate(fitted.transform(test_df))
    fi = get_feature_importance(fitted)
    mlflow.log_param('model', 'RandomForest')
    mlflow.log_param('numTrees', 50)
    mlflow.log_metrics(metrics)
    if fi: mlflow.log_dict(fi, 'feature_importance.json')
    print(f'  RMSE={metrics["rmse"]}  MAE={metrics["mae"]}  R2={metrics["r2"]}')
    results.append({'model': 'RandomForest', **metrics, 'feature_importance': fi, 'fitted': fitted})

# gradient boosted trees
print('[ML] GBT egitiliyor...')
with mlflow.start_run(run_name='GBT'):
    model = GBTRegressor(labelCol=TARGET_COL, featuresCol='features', maxIter=20, seed=42)
    fitted = build_pipeline(model).fit(train_df)
    metrics = evaluate(fitted.transform(test_df))
    fi = get_feature_importance(fitted)
    mlflow.log_param('model', 'GBT')
    mlflow.log_param('maxIter', 20)
    mlflow.log_metrics(metrics)
    if fi: mlflow.log_dict(fi, 'feature_importance.json')
    print(f'  RMSE={metrics["rmse"]}  MAE={metrics["mae"]}  R2={metrics["r2"]}')
    results.append({'model': 'GBT', **metrics, 'feature_importance': fi, 'fitted': fitted})

# genellestirilmis lineer regresyon
print('[ML] GeneralizedLinearReg egitiliyor...')
with mlflow.start_run(run_name='GeneralizedLinearReg'):
    model = GeneralizedLinearRegression(labelCol=TARGET_COL, featuresCol='features',
                                        family='gaussian', link='identity')
    fitted = build_pipeline(model).fit(train_df)
    metrics = evaluate(fitted.transform(test_df))
    mlflow.log_param('model', 'GeneralizedLinearReg')
    mlflow.log_metrics(metrics)
    print(f'  RMSE={metrics["rmse"]}  MAE={metrics["mae"]}  R2={metrics["r2"]}')
    results.append({'model': 'GeneralizedLinearReg', **metrics, 'feature_importance': {}, 'fitted': fitted})

[ML] RandomForest egitiliyor...


  RMSE=3.5129  MAE=2.7184  R2=0.867
[ML] GBT egitiliyor...


  RMSE=2.9077  MAE=2.2113  R2=0.9089
[ML] GeneralizedLinearReg egitiliyor...


26/05/12 14:55:12 WARN Instrumentation: [998c40b0] regParam is zero, which might cause numerical instability and overfitting.


  RMSE=2.9714  MAE=2.27  R2=0.9049


## En İyi Model — MLflow Kayıt

In [5]:
# rmse ye gore en iyi modeli sec ve kaydet
best = min(results, key=lambda x: x['rmse'])
print(f'[ML] En iyi model: {best["model"]}  (RMSE={best["rmse"]})')

os.makedirs(MODEL_PATH, exist_ok=True)
best['fitted'].write().overwrite().save(MODEL_PATH)
print(f'[ML] Model kaydedildi -> {MODEL_PATH}')

summary = [{k: v for k, v in r.items() if k not in ('fitted', 'feature_importance')}
           for r in results]
pd.DataFrame(summary).to_csv('./mlflow_data/model_comparison.csv', index=False)
print('[ML] Karsilastirma tablosu kaydedildi.')

fi_data = {r['model']: r['feature_importance'] for r in results if r['feature_importance']}
with open('./mlflow_data/feature_importance.json', 'w') as f:
    json.dump(fi_data, f, indent=2)
print('[ML] Feature importance kaydedildi.')

[ML] En iyi model: GBT  (RMSE=2.9077)
[ML] Model kaydedildi -> ./mlflow_data/best_model
[ML] Karsilastirma tablosu kaydedildi.
[ML] Feature importance kaydedildi.


In [6]:
print('\n[OZET] Model Karsilastirma:')
for r in sorted(results, key=lambda x: x['rmse']):
    print(f"  {r['model']:30s}  RMSE={r['rmse']}  MAE={r['mae']}  R2={r['r2']}")

spark.stop()
print('\nML pipeline tamamlandi.')


[OZET] Model Karsilastirma:
  GBT                             RMSE=2.9077  MAE=2.2113  R2=0.9089
  LinearRegression                RMSE=2.9714  MAE=2.27  R2=0.9049
  GeneralizedLinearReg            RMSE=2.9714  MAE=2.27  R2=0.9049
  DecisionTree                    RMSE=3.103  MAE=2.3579  R2=0.8963
  RandomForest                    RMSE=3.5129  MAE=2.7184  R2=0.867

ML pipeline tamamlandi.
